In [ ]:
import torch
import os
print("PyTorch has version {}".format(torch.__version__))

PyTorch has version 2.10.0+cu128


In [ ]:
!pip uninstall -y torch-geometric torch-sparse torch-scatter torch-cluster torch-spline-conv pyg-lib

torch_version = str(torch.__version__)
scatter_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
sparse_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
pyg_lib_src = f"https://data.pyg.org/whl/torch-{torch_version}.html"
!pip install torch-scatter -f $scatter_src
!pip install torch-sparse -f $sparse_src
!pip install pyg-lib -f $pyg_lib_src
!pip install torch-geometric
!pip install ogb

Looking in links: https://pytorch-geometric.com/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.4 MB/s eta 0:00:00
Looking in links: https://pytorch-geometric.com/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 54.7 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 4.0 MB/s eta 0:00:00


### Imports

In [ ]:
import ast
from torch_geometric.data import HeteroData
from google.colab import drive
from torch import Tensor
import torch_geometric.transforms as T
from torch_geometric.utils import coalesce
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import SAGEConv, to_hetero
import torch.nn.functional as F
import tqdm
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)


### Load the data and create the graph

In [ ]:
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Colab/australian_users_items.json'

def create_pyg_graph(file_path):
    data = HeteroData()

    user_map = {}
    item_map = {}

    item_names = []

    sources = []
    targets = []

    with open(file_path, encoding='utf-8') as f:
      for line in f:
          user = ast.literal_eval(line)
          u_id = str(user['user_id'])

          if u_id not in user_map:
              user_map[u_id] = len(user_map)
          u_idx = user_map[u_id]

          for item in user['items']:
              i_id = str(item.get('item_id'))

              if i_id not in item_map:
                  item_map[i_id] = len(item_map)
                  item_names.append(item.get('item_name', 'Unknown'))

              i_idx = item_map[i_id]

              sources.append(u_idx)
              targets.append(i_idx)

    num_users = len(user_map)
    num_items = len(item_map)

    data['user'].num_nodes = num_users
    data['item'].num_nodes = num_items

    data['user'].node_id = torch.arange(num_users)
    data['item'].node_id = torch.arange(num_items)

    edge_index = torch.tensor([sources, targets], dtype=torch.long)
    edge_index = coalesce(edge_index)
    data['user', 'purchased', 'item'].edge_index = edge_index

    data = T.ToUndirected()(data)

    return data, item_names, user_map, item_map

data, item_names, user_map, item_map = create_pyg_graph(file_path)

Mounted at /content/drive


### Train test split

In [ ]:
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.3,
    neg_sampling_ratio=2.0,
    add_negative_train_samples=False,
    edge_types=("user", "purchased", "item"),
    rev_edge_types=("item", "rev_purchased", "user"),
)
train_data, val_data, test_data = transform(data)

edge_label_index = train_data["user", "purchased", "item"].edge_label_index
edge_label = train_data["user", "purchased", "item"].edge_label

train_loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[20, 10],
    neg_sampling_ratio=2.0,
    edge_label_index=(("user", "purchased", "item"), edge_label_index),
    edge_label=edge_label,
    batch_size=128,
    shuffle=True,
)


### Creating the GNN model

In [ ]:
class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()

        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

class Classifier(torch.nn.Module):
    def forward(self, x_user: Tensor, x_item: Tensor, edge_label_index: Tensor) -> Tensor:
        edge_feat_user = x_user[edge_label_index[0]]
        edge_feat_item = x_item[edge_label_index[1]]

        return (edge_feat_user * edge_feat_item).sum(dim=-1)


class Model(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.item_lin = torch.nn.Linear(20, hidden_channels)
        self.user_emb = torch.nn.Embedding(data["user"].num_nodes, hidden_channels)
        self.item_emb = torch.nn.Embedding(data["item"].num_nodes, hidden_channels)

        self.gnn = GNN(hidden_channels)
        self.gnn = to_hetero(self.gnn, metadata=data.metadata())

        self.classifier = Classifier()

    def forward(self, data: HeteroData) -> Tensor:
        x_dict = {
          "user": self.user_emb(data["user"].node_id),
          "item": self.item_emb(data["item"].node_id)
        }

        x_dict = self.gnn(x_dict, data.edge_index_dict)
        pred = self.classifier(
            x_dict["user"],
            x_dict["item"],
            data["user", "purchased", "item"].edge_label_index,
        )

        return pred

### Training the model

In [ ]:
model = Model(hidden_channels=64)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: '{device}'")

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1, 6):
    total_loss = total_examples = 0
    model.train()
    for sampled_data in tqdm.tqdm(train_loader):
        optimizer.zero_grad()

        sampled_data.to(device)
        pred = model(sampled_data)

        ground_truth = sampled_data["user", "purchased", "item"].edge_label
        loss = F.binary_cross_entropy_with_logits(pred, ground_truth)

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * pred.numel()
        total_examples += pred.numel()
    print(f"Epoch: {epoch:03d}, Loss: {total_loss / total_examples:.4f}")

Device: 'cuda'


100%|██████████| 9552/9552 [03:10<00:00, 50.15it/s]


Epoch: 001, Loss: 0.2156


100%|██████████| 9552/9552 [03:03<00:00, 51.97it/s]


Epoch: 002, Loss: 0.1869


...

Epoch: 003, Loss: 0.1787


100%|██████████| 9552/9552 [03:04<00:00, 51.89it/s]


Epoch: 004, Loss: 0.1731


100%|██████████| 9552/9552 [03:03<00:00, 52.17it/s]

Epoch: 005, Loss: 0.1690


### Evaluating the model (simple metrics)

In [ ]:
model.eval()
edge_label_index = val_data["user", "purchased", "item"].edge_label_index
edge_label = val_data["user", "purchased", "item"].edge_label

val_loader = LinkNeighborLoader(
    data=val_data,
    num_neighbors=[20, 10],
    edge_label_index=(("user", "purchased", "item"), edge_label_index),
    edge_label=edge_label,
    batch_size=3 * 128,
    shuffle=False,
)


preds = []
ground_truths = []

model.eval()

for sampled_data in tqdm.tqdm(val_loader):
    with torch.no_grad():
        sampled_data = sampled_data.to(device)

        pred = model(sampled_data)

        preds.append(pred.cpu())
        ground_truths.append(
            sampled_data["user", "purchased", "item"].edge_label.cpu()
        )

pred = torch.cat(preds, dim=0)
ground_truth = torch.cat(ground_truths, dim=0)

pred_prob = torch.sigmoid(pred).numpy()
pred_binary = (pred_prob >= 0.5).astype(int)
ground_truth = ground_truth.numpy()

auc = roc_auc_score(ground_truth, pred_prob)
precision = precision_score(ground_truth, pred_binary)
recall = recall_score(ground_truth, pred_binary)
f1 = f1_score(ground_truth, pred_binary)

print()
print(f"Validation AUC:       {auc:.4f}")
print(f"Validation Precision: {precision:.4f}")
print(f"Validation Recall:    {recall:.4f}")
print(f"Validation F1-score:  {f1:.4f}")

100%|██████████| 3980/3980 [00:55<00:00, 72.26it/s]



Validation AUC:       0.9801
Validation Precision: 0.8933
Validation Recall:    0.9061
Validation F1-score:  0.8997


### Evaluating the model (Recall at K)

In [ ]:
@torch.no_grad()
def global_recall_at_k(model, data, val_data, k=20):
    model.eval()
    device = next(model.parameters()).device

    edge_index_dict = {k: v.to(device) for k, v in data.edge_index_dict.items()}

    x_dict = {
        "user": model.user_emb(data["user"].node_id.to(device)),
        "item": model.item_emb(data["item"].node_id.to(device))
    }

    x_dict = model.gnn(x_dict, edge_index_dict)

    user_feats = x_dict["user"]
    item_feats = x_dict["item"]

    edge_index = val_data["user", "purchased", "item"].edge_label_index.cpu()
    edge_label = val_data["user", "purchased", "item"].edge_label.cpu()
    pos_edges = edge_index[:, edge_label == 1]

    hits = 0
    unique_users = pos_edges[0].unique()

    for u_idx in tqdm.tqdm(unique_users, desc="Global Recall@K"):
        ground_truth_items = pos_edges[1, pos_edges[0] == u_idx].tolist()

        u_feat = user_feats[u_idx].unsqueeze(0)
        scores = torch.matmul(u_feat, item_feats.t()).squeeze()

        _, top_k_items = torch.topk(scores, k=k)
        top_k_items = top_k_items.cpu().tolist()

        for item in ground_truth_items:
            if item in top_k_items:
                hits += 1

    return hits / pos_edges.size(1)

recall_at_k = global_recall_at_k(model, data, val_data, k=20)
print()
print(f"Global Recall@20: {recall_at_k:.4f}")

Global Recall@K: 100%|██████████| 58945/58945 [01:58<00:00, 498.74it/s]


Global Recall@20: 0.1243


### Making predictions for a user

In [ ]:
@torch.no_grad()
def make_predictions_for_a_user(user_id_str, model, data, user_map, item_map, item_names, k=10):
    model.eval()
    device = next(model.parameters()).device

    if user_id_str not in user_map:
        return f"User '{user_id_str}' not found."
    u_idx = user_map[user_id_str]

    edge_index = data['user', 'purchased', 'item'].edge_index
    current_item_indices = edge_index[1, edge_index[0] == u_idx].tolist()
    current_games = [item_names[i] for i in current_item_indices]

    edge_index_dict = {k: v.to(device) for k, v in data.edge_index_dict.items()}
    x_dict = {
        "user": model.user_emb(data["user"].node_id.to(device)),
        "item": model.item_emb(data["item"].node_id.to(device))
    }
    x_dict = model.gnn(x_dict, edge_index_dict)

    u_feat = x_dict["user"][u_idx].unsqueeze(0)
    item_feats = x_dict["item"]
    scores = torch.matmul(u_feat, item_feats.t()).squeeze()

    _, top_indices = torch.topk(scores, k=k + len(current_item_indices))
    top_indices = top_indices.cpu().tolist()

    recommendations = []
    for idx in top_indices:
        if idx not in current_item_indices:
            recommendations.append(item_names[idx])
        if len(recommendations) == k:
            break

    print(f"=== User: {user_id_str} ===")
    print(f"Current Library ({len(current_games)} games):")
    for i, game in enumerate(current_games[:30], 1):
        print(f"  - {game}")
    if len(current_games) > 30: print(f"  ... and {len(current_games)-30} more.")

    print(f"\nTop {k} Recommendations (New to User):")
    for i, game in enumerate(recommendations, 1):
        print(f"  {i}. {game}")
    print("=" * 40)

user_id = list(user_map.keys())[30]
make_predictions_for_a_user(user_id, model, data, user_map, item_map, item_names, k=10)

=== User: fdcfelipefdc ===
Current Library (43 games):
  - Borderlands
  - Dota 2 Test
  - Counter-Strike: Global Offensive
  - War Thunder
  - PAYDAY 2
  - Insurgency
  - Dead Island: Epidemic
  - Evolve Stage 2
  - Garry's Mod
  - Age of Empires® III: Complete Collection
  - Castle Crashers
  - Quake Live
  - Magicka: Wizard Wars
  - Sniper Elite V2
  - PAYDAY: The Heist
  - Transformice
  - ARK: Survival Evolved
  - ARK: Survival Of The Fittest
  - Immune
  - Warframe
  - Path of Exile
  - No More Room in Hell
  - Rocket League
  - Bloody Trapland
  - Unturned
  - RaceRoom Racing Experience 
  - Aura Kingdom
  - Robocraft
  - TERA
  - Grand Theft Auto V
  ... and 13 more.

Top 10 Recommendations (New to User):
  1. Heroes & Generals
  2. PlanetSide 2
  3. Left 4 Dead 2
  4. Terraria
  5. Dirty Bomb
  6. Left 4 Dead 2 Beta
  7. Loadout
  8. Warface
  9. Blacklight: Retribution
  10. Fistful of Frags
